In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/haseebekhan/healthcare-cost-dataset/Healthcare Cost Dataset/sample_submission.csv
/kaggle/input/datasets/haseebekhan/healthcare-cost-dataset/Healthcare Cost Dataset/test/test/main_df_test.csv
/kaggle/input/datasets/haseebekhan/healthcare-cost-dataset/Healthcare Cost Dataset/test/test/drg_df_test.csv
/kaggle/input/datasets/haseebekhan/healthcare-cost-dataset/Healthcare Cost Dataset/test/test/cpt_df_test.csv
/kaggle/input/datasets/haseebekhan/healthcare-cost-dataset/Healthcare Cost Dataset/test/test/icd_df_test.csv
/kaggle/input/datasets/haseebekhan/healthcare-cost-dataset/Healthcare Cost Dataset/test/test/dob_df.csv
/kaggle/input/datasets/haseebekhan/healthcare-cost-dataset/Healthcare Cost Dataset/train/train/main_df_train.csv
/kaggle/input/datasets/haseebekhan/healthcare-cost-dataset/Healthcare Cost Dataset/train/train/cpt_df_train.csv
/kaggle/input/datasets/haseebekhan/healthcare-cost-dataset/Healthcare Cost Dataset/train/train/drg_df_train.csv
/kaggle/input/dat

# **Binary Classification**

## **Preprocessing**

In [2]:
!pip install deslib


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.6/172.6 kB 3.2 MB/s eta 0:00:00a 0:00:01


In [3]:
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import VotingClassifier
from deslib.des.knora_e import KNORAE # (Stacking)
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from typing import Tuple, List, Optional

import xgboost as xgb
import lightgbm as lgb
import catboost as cgb

from sklearn.metrics import accuracy_score, mean_squared_error, f1_score, confusion_matrix, ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import validation_curve


import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

import warnings

warnings.filterwarnings("ignore")


### **PATHS**


In [4]:
BASE_DIR = r"/kaggle/input/datasets/haseebekhan/healthcare-cost-dataset/Healthcare Cost Dataset"

TRAINING_PATH = os.path.join(BASE_DIR, "train", "train") + os.sep
TESTING_PATH  = os.path.join(BASE_DIR, "test",  "test")  + os.sep

COST_THRESHOLD = 30_000 # Cost which split it into binary label
N_FOLDS        = 5
RANDOM_STATE   = 42

In [5]:
 main_df = pd.read_csv(TRAINING_PATH + "main_df_train.csv", low_memory=False)

In [6]:
main_df.count()

Member_Key            807420
YEAR                  807420
MONTH                 807420
ProviderVisitCount    807420
ProviderCharges       807420
                       ...  
Admission             807420
AdmissionCost         807420
AdmissionLOS          807420
NEXT_YEAR_COST        807420
HighCostLabel         807420
Length: 301, dtype: int64

In [7]:
def compute_vif(df: pd.DataFrame) -> pd.DataFrame:
    """VIF for every numeric column. Constant columns are dropped first --
    a constant column gives a meaningless inf/NaN VIF that has nothing to
    do with collinearity against other features."""
    X = df.select_dtypes(include=[np.number]).copy()
    X = X.loc[:, X.nunique(dropna=False) > 1]
    X = sm.add_constant(X, has_constant="add")
    vif_df = pd.DataFrame({
        "feature": X.columns,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    })
    return vif_df[vif_df["feature"] != "const"].sort_values("VIF", ascending=False).reset_index(drop=True)


def fit_vif_drop_list(X_train: pd.DataFrame, exclude_cols=None,
                       vif_threshold: float = 10.0, max_iter: int = 200) -> list:
    """
    Iteratively drops the single worst-VIF column, recomputes, repeats --
    NOT a one-shot 'drop everything with VIF==inf', because removing one
    column changes every other column's VIF. Fit on TRAIN ONLY; apply the
    resulting list unchanged to val/test/submission via drop_multicollinear().
    """
    exclude_cols = set(exclude_cols or [])
    candidates = [c for c in X_train.select_dtypes(include=[np.number]).columns
                  if c not in exclude_cols]
    working = X_train[candidates].copy()
    dropped = []

    for _ in range(max_iter):
        nunique = working.nunique(dropna=False)
        const_cols = nunique[nunique <= 1].index.tolist()
        if const_cols:
            dropped.extend(const_cols)
            working = working.drop(columns=const_cols)
            continue

        if working.shape[1] <= 1:
            break

        vif_df = compute_vif(working)
        worst = vif_df.iloc[0]
        if worst["VIF"] <= vif_threshold:
            break

        dropped.append(worst["feature"])
        working = working.drop(columns=[worst["feature"]])

    print(f"    [VIF] dropping {len(dropped)} columns at threshold {vif_threshold}: {dropped}")
    return dropped


def drop_multicollinear(df: pd.DataFrame, drop_cols: list) -> pd.DataFrame:
    """Apply a pre-fit (train-only) drop list to any split."""
    return df.drop(columns=[c for c in drop_cols if c in df.columns])

In [8]:
"""
Design principles (why this differs from the original):

1. MISSING != ZERO. For utilization/cost columns in claims data, NaN usually
   does mean "no event happened" -> 0 is correct. But for a handful of
   columns (days-since-last-visit, HCC risk scores, anything paired with an
   eligibility flag) NaN means "we don't know" or "not applicable", and
   filling with 0 actively lies to the model. Those get sentinel values +
   an explicit `_was_missing` indicator instead.

2. NEGATIVE != INVALID everywhere. For *_Cost columns, negative numbers in
   claims data are frequently legitimate (refund/adjustment/reversal lines),
   not data errors. Silently clipping them to 0 throws away a real signal
   (a member with lots of cost reversals looks different from one with
   none). For counts/LOS/HCC scores, negative truly is impossible, so those
   ARE treated as data errors.

3. NO LEAKAGE. Every statistic used to impute (medians, frequency maps,
   sentinel values, winsorization bounds) is learned with `fit_main_stats()`
   on the TRAINING data only, then applied to both train and test with
   `transform_main()`. The original function recomputed stats independently
   on whatever df was passed in -- if you call it separately on train and
   test you get two different imputation rules, which is its own subtle
   form of leakage/inconsistency.

4. Conditional columns are imputed conditionally. AWV_Compliant only makes
   sense if AWV_Eligible == 1. Collapsing "not eligible" and "eligible but
   missing compliance data" into the same 0 (as the original code does)
   destroys that distinction.

Usage:
    stats = fit_main_stats(train_df)
    train_clean = transform_main(train_df, stats)
    test_clean  = transform_main(test_df, stats)   # same stats, no refitting
"""

# ---------------------------------------------------------------------------
# Static column metadata
# ---------------------------------------------------------------------------

LEAKAGE_COLS = [
    "MONTH", "ActualYear", "ActualMonthNumber", "IsYTD", "QUARTER",
    "PredictedCost", "Leakage_Cost", "IsLatest",
]

DATE_COLS = [("AWVLastDate", "DaysSinceAWV"), ("LastPCPVisit", "DaysSinceLastPCP")]

# Simple flags with no conditional partner -> NaN really does mean "false"
SIMPLE_BINARY_COLS = [
    "IsCovid", "DoneBySelf", "AWV", "Admission", "Ishighrisk",
    "NonClaimBasedPayment", "AWV_Compliant_Eligible",
]

# (eligibility_col, dependent_col) pairs -> dependent only meaningful if eligible
ELIGIBLE_DEPENDENT_PAIRS = [
    ("AWV_Eligible", "AWV_Compliant"),
    ("IPPE_Eligible", "IPPE_Compliant"),
]

NETWORK_PAIR = ("InNetworkPCP", "OutNetworkPCP")  # -> derive "NoPCPOnRecord"

DISEASE_COLS = [
    "ChronicObstructivePulmonaryDiseaseOrAsthma", "CongestiveHeartFailure",
    "BacterialPneumonia", "DiabetesShortTermComplications",
    "DiabetesLongTermComplications", "UncontrolledDiabetes",
    "AmputationDiabetes", "Dialysis", "ChronicConditions",
]

HCC_COLS = [
    "MemberHccScore", "MemberHccScoreLastYear",
    "MemberHccScoreLastTwoYear", "PersiviaMemberHccScore",
]

CAT_COLS = [
    "AWVCode", "AWVStatus", "AttributionStatus", "LastPCPProvider",
    "AWVProviderNetwork", "Payers_key", "Enrollment_key",
]

COUNT_COL_CANDIDATES = [
    "ProviderVisitCount", "ERAdmisison", "30dayHospitalReadmission", "OutpatientVisits",
    "Inpatient", "HomeHealth", "Hospice", "SkilledNursingFacilities",
    "EmergencyDepartmentVisits", "EmergencyDepartmentVisitsWithAdmissions",
    "CTEvents", "MRIEvents", "RadiologyEvents", "OtherImagingServices",
    "LabEventsPathalogy", "LabEventsClinicalDiagnostics", "PrimaryCareServicesTotal",
    "PrimaryCareServiceswithPrimaryCarePhysician", "PrimaryCareServicesWithSpecialistPhysician",
    "PrimaryCareServicesWithNursePractitioner", "30DayReadmission", "NewPatients",
    "EstablishedPatients", "PostDischargeVisits", "AmbulanceEvents", "Medication",
    "PCPVisits", "OfficeVisits", "EDVisitsWithNoFollowUp", "IPVisitsWithNoFollowUp",
    "TotalClaims", "TotalScriptsFilled", "AWVVisits", "PediatricsVisits", "UrgentCareVisit",
    "AmbulatorySurgeryVisit", "DentistEvents",
]

MISSING_FLAG_THRESHOLD = 0.01  # add a `_was_missing` indicator if >1% missing



# ---------------------------------------------------------------------------
# FIT: learn every statistic from TRAINING data only
# ---------------------------------------------------------------------------

def fit_main_stats(train_df: pd.DataFrame) -> dict:
    
    df = train_df[train_df["MONTH"] == -1] if "MONTH" in train_df.columns else train_df
    
    cost_cols = [c for c in df.columns if c.endswith("_Cost") and c != "TotalCost"]
    los_cols = [c for c in df.columns if "LOS" in c]
    count_cols = [c for c in COUNT_COL_CANDIDATES if c in df.columns]

    stats = {"cost_cols": cost_cols, "los_cols": los_cols, "count_cols": count_cols}

    # --- Costs: winsorization bounds, NOT a hard clip-to-zero.
    # Lower bound allows legitimate negative adjustments through; only the
    # extreme tail (likely true data errors) gets capped.
    
    stats["cost_bounds"] = {}
    for c in cost_cols:
        col = df[c].dropna()
        if len(col):
            lo = min(0, col.quantile(0.001))   # keep small negatives, cap extreme ones
            hi = col.quantile(0.995)
            stats["cost_bounds"][c] = (lo, hi)
        else:
            stats["cost_bounds"][c] = (0, 0)

    # --- LOS: median of POSITIVE values only, used to repair negative/invalid entries
    stats["los_median"] = { c: (df.loc[df[c] > 0, c].median() if (df[c] > 0).any() else 0.0) for c in los_cols }

    # --- HCC scores: median grouped by Ishighrisk (falls back to global median)
    stats["hcc_global_median"] = {}
    stats["hcc_group_median"] = {}
    
    for c in HCC_COLS:
        if c in df.columns:
            valid = df[c].where(df[c] >= 0)  # negative HCC score is invalid -> NaN
            stats["hcc_global_median"][c] = valid.median()
            if "Ishighrisk" in df.columns:
                stats["hcc_group_median"][c] = valid.groupby(df["Ishighrisk"]).median().to_dict()

    # --- Categorical: frequency map learned on TRAIN ONLY, with a fallback
    # for categories never seen in training (e.g. new provider codes in 2024)
    stats["cat_freq"] = {}
    stats["cat_freq_fallback"] = {}
    for c in CAT_COLS:
        if c in df.columns:
            vc = df[c].astype(str).fillna("Unknown").value_counts(normalize=True)
            stats["cat_freq"][c] = vc.to_dict()
            stats["cat_freq_fallback"][c] = vc.min() if len(vc) else 0.0

    # --- Date-derived "days since X": sentinel for "never happened",
    # learned as (max observed gap + 1 year) so it sits clearly past the
    # real range rather than colliding with an arbitrary 0.
    stats["date_sentinel"] = {}
    ref_date = pd.Timestamp("2023-01-01")
    for raw_col, new_col in DATE_COLS:
        if raw_col in df.columns:
            days = (ref_date - pd.to_datetime(df[raw_col], errors="coerce")).dt.days
            days = days.clip(lower=0)
            max_seen = days.max()
            stats["date_sentinel"][new_col] = (max_seen if pd.notna(max_seen) else 0) + 365

    # --- Per-column missing rate (train), used to decide whether a
    # `_was_missing` indicator is worth adding for "simple" numeric columns
    stats["missing_rate"] = df.isna().mean().to_dict()

    # --- Global numeric median fallback, for any column not covered above
    # (kept separate from the original's blanket fillna(0) so unexpected
    # columns get a sane default instead of being silently zeroed)
    stats["global_median"] = df.select_dtypes(include=[np.number]).median().to_dict()

    return stats


# ---------------------------------------------------------------------------
# TRANSFORM: apply stats (from fit_main_stats) to ANY split (train or test)
# ---------------------------------------------------------------------------
def aggregate_main(df: pd.DataFrame, stats: dict) -> pd.DataFrame:
    df = df.copy()

    if "MONTH" in df.columns:
        df = df[df["MONTH"] == -1].reset_index(drop=True)

    df.drop(columns=[c for c in LEAKAGE_COLS if c in df.columns], inplace=True)

    # ---- Dates: sentinel for "never happened" + explicit indicator ----
    ref_date = pd.Timestamp("2023-01-01")
    for raw_col, new_col in DATE_COLS:
        if raw_col in df.columns:
            parsed = pd.to_datetime(df[raw_col], errors="coerce")
            gap = (ref_date - parsed).dt.days.clip(lower=0)
            never_flag = gap.isna()
            sentinel = stats["date_sentinel"].get(new_col, 9999)
            df[new_col] = gap.fillna(sentinel)
            df[f"{new_col}_NeverRecorded"] = never_flag.astype(int)
            df.drop(columns=[raw_col], inplace=True)

    cost_cols = [c for c in stats["cost_cols"] if c in df.columns]
    los_cols = [c for c in stats["los_cols"] if c in df.columns]
    count_cols = [c for c in stats["count_cols"] if c in df.columns]

    # ---- Costs: NaN -> 0 (no claim), but winsorize instead of clipping
    # negatives to zero, so legitimate refunds/adjustments survive. ----
    for c in cost_cols:
        df[c] = df[c].fillna(0)
        lo, hi = stats["cost_bounds"].get(c, (0, None))
        df[c] = df[c].clip(lower=lo, upper=hi)

    # ---- Counts: NaN -> 0 (no events); negative is impossible -> flag + zero ----
    for c in count_cols:
        invalid = df[c] < 0
        if invalid.any():
            df[f"{c}_HadInvalidNegative"] = invalid.astype(int)
        df[c] = df[c].fillna(0).clip(lower=0)

    # ---- LOS: NaN -> 0 (no stay of that type). Negative is a date-order
    # data error -> repair with the train-derived positive median instead
    # of zeroing (zero would understate a real, unknown stay length). ----
    for c in los_cols:
        invalid = df[c] < 0
        if invalid.any():
            df[f"{c}_WasInvalid"] = invalid.astype(int)
            df.loc[invalid, c] = stats["los_median"].get(c, 0.0)
        df[c] = df[c].fillna(0).clip(lower=0)

    # ---- Simple binaries: NaN -> 0, but add a missingness indicator when
    # missingness is non-trivial, since "we don't know" can differ from
    # "false" in claims data. ----
    for c in SIMPLE_BINARY_COLS:
        if c in df.columns:
            miss_rate = stats["missing_rate"].get(c, 0)
            if miss_rate > MISSING_FLAG_THRESHOLD:
                df[f"{c}_WasMissing"] = df[c].isna().astype(int)
            df[c] = df[c].fillna(0).replace({True: 1, False: 0}).astype(int)

    # ---- Eligible/Compliant pairs: only impute "compliant" conditional on
    # eligibility, and keep "ineligible" distinguishable from "eligible but
    # data missing". ----
    for elig_col, dep_col in ELIGIBLE_DEPENDENT_PAIRS:
        if elig_col in df.columns and dep_col in df.columns:
            elig = df[elig_col].fillna(0).replace({True: 1, False: 0}).astype(int)
            dep_missing_while_eligible = (elig == 1) & df[dep_col].isna()
            df[f"{dep_col}_MissingWhileEligible"] = dep_missing_while_eligible.astype(int)
            dep = df[dep_col].fillna(0).replace({True: 1, False: 0}).astype(int)
            dep = np.where(elig == 0, 0, dep)  # can't be compliant if ineligible
            df[elig_col] = elig
            df[dep_col] = dep

    # ---- PCP network pair: distinguish "no PCP on record" from "in network" ----
    in_col, out_col = NETWORK_PAIR
    if in_col in df.columns and out_col in df.columns:
        no_pcp = df[in_col].isna() & df[out_col].isna()
        df["NoPCPOnRecord"] = no_pcp.astype(int)
        df[in_col] = df[in_col].fillna(0).astype(int)
        df[out_col] = df[out_col].fillna(0).astype(int)

    # ---- Disease/condition flags: NaN -> 0 (condition not coded = treated
    # as absent, standard convention for claims/diagnosis flags). ----
    for c in DISEASE_COLS:
        if c in df.columns:
            df[c] = df[c].fillna(0).astype(int) if df[c].dropna().isin([0, 1, True, False]).all() else df[c].fillna(0)

    # ---- HCC risk scores: negative is invalid -> NaN, then impute by
    # Ishighrisk-conditioned median (train-derived), not a blind global
    # median. Always keep a missingness flag: absence of a risk score is
    # itself informative (e.g. brand-new enrollee, no claims history yet). ----
    for c in HCC_COLS:
        if c in df.columns:
            df[f"{c}_WasMissing"] = df[c].isna().astype(int)
            df[c] = df[c].where(df[c] >= 0)  # drop invalid negatives -> NaN
            if "Ishighrisk" in df.columns and c in stats.get("hcc_group_median", {}):
                group_map = stats["hcc_group_median"][c]
                fallback = stats["hcc_global_median"].get(c, 0.0)
                df[c] = df[c].fillna(df["Ishighrisk"].map(group_map)).fillna(fallback)
            else:
                df[c] = df[c].fillna(stats["hcc_global_median"].get(c, 0.0))

    # ---- Categorical: frequency-encode using TRAIN-only map; unseen
    # categories (e.g. new provider in 2024 test set) get the train
    # minimum frequency rather than NaN or an inflated guess. ----
    for c in CAT_COLS:
        if c in df.columns:
            freq_map = stats["cat_freq"].get(c, {})
            fallback = stats["cat_freq_fallback"].get(c, 0.0)
            df[c] = df[c].astype(str).fillna("Unknown").map(freq_map).fillna(fallback)

    # ---- Anything left over: train-column median (not a blanket 0), and
    # explicitly surface what was caught here so unexpected schema changes
    # don't get silently zeroed. ----
    num_cols = df.select_dtypes(include=[np.number]).columns
    leftover_na = df[num_cols].isna().sum()
    leftover_cols = leftover_na[leftover_na > 0].index.tolist()
    if leftover_cols:
        print(f"    [process_main_v2] filling leftover NaNs with train median for: {leftover_cols}")
        for c in leftover_cols:
            df[c] = df[c].fillna(stats["global_median"].get(c, 0.0))

    print(f"    main_df → {df.shape[0]:,} rows × {df.shape[1]} cols")
    return df


"""
Fixes applied to the original dob_df / cpt_df / drg_df / icd_df helpers:

1. Age is no longer frozen at a single `reference_year`. dob processing now
   only cleans Gender_Key/AgeGroup_Key and produces a `BirthYear`. Age is
   computed later, per row, against that row's own YEAR column in main_df
   (`add_age()`), so a member's age correctly changes across the years in
   the panel instead of being stuck at their 2023 age (which, for any row
   before 2023, was a straight temporal leak).

2. cpt/icd grouping now explicitly extracts the calendar year from the
   date column via `_to_year()` instead of grouping on a raw date and
   relabeling it "YEAR" afterwards. (drg already had a pre-extracted
   `Start_Date_Year`, hence why it didn't show this bug.)

3. Codes are NaN-filled with an explicit "Unknown" BEFORE casting to str,
   so a missing code never silently becomes the literal string "nan" and
   contaminates the mode()/frequency calculations.

4. All frequency encodings (cpt_TopProcedure, drg_MostCommon, icd_TopICD)
   and the DRG rare-code set are now learned ONCE from training data via
   fit_*_stats(), then applied unchanged to both train and test via the
   aggregate_*() functions -- mirroring process_main_v2.py's fit/transform
   split. Previously "rare" was judged using counts that could include the
   2024 holdout year, which is leakage.

Usage:
    dob_stats  = fit_dob_stats(dob_train)
    cpt_stats  = fit_cpt_stats(cpt_train)
    drg_stats  = fit_drg_stats(drg_train)
    icd_stats  = fit_icd_stats(icd_train)

    dob_clean  = process_dob(dob_df, dob_stats)          # same df works for train+test
    cpt_agg    = aggregate_cpt(cpt_train_or_test, cpt_stats)
    drg_agg    = aggregate_drg(drg_train_or_test, drg_stats)
    icd_agg    = aggregate_icd(icd_train_or_test, icd_stats)

    main_df = add_age(main_df, dob_clean)                # merge + compute per-row age
"""

def _to_year(series: pd.Series) -> pd.Series:
    """Return calendar year whether `series` is already a year (int-like)
    or a full date/timestamp. Avoids the StartDate-mislabeled-as-YEAR bug."""
    if pd.api.types.is_numeric_dtype(series):
        return series.astype("Int64")
    return pd.to_datetime(series, errors="coerce").dt.year


# ═══════════════════════════════════════════════════════════════════════════
#  dob_df
# ═══════════════════════════════════════════════════════════════════════════
def fit_dob_stats(dob_train: pd.DataFrame) -> dict:
    dob = dob_train.copy()
    birth_year = pd.to_datetime(dob["DOB_Key"], errors="coerce").dt.year
    return {
        "birth_year_median": birth_year.median(),
        "gender_mode": dob["Gender_Key"].mode().iloc[0] if dob["Gender_Key"].notna().any() else 0,
        "agegroup_mode": dob["AgeGroup_Key"].mode().iloc[0] if dob["AgeGroup_Key"].notna().any() else 0,
    }


def aggregate_dob(dob: pd.DataFrame, stats: dict) -> pd.DataFrame:
    """Cleans dob_df but deliberately does NOT compute a final Age column --
    age depends on which YEAR row it's joined to in main_df, so it's
    computed later in add_age()."""
    dob = dob.copy()
    dob["BirthYear"] = pd.to_datetime(dob["DOB_Key"], errors="coerce").dt.year
    dob["BirthYear"] = dob["BirthYear"].fillna(stats["birth_year_median"])
    dob["Gender_Key"] = dob["Gender_Key"].fillna(stats["gender_mode"]).astype(int)
    dob["AgeGroup_Key"] = dob["AgeGroup_Key"].fillna(stats["agegroup_mode"]).astype(int)
    dob.drop(columns=["DOB_Key"], inplace=True)
    print(f"    dob_df  → {dob.shape[0]:,} rows × {dob.shape[1]} cols")
    return dob


def add_age(main_df: pd.DataFrame, dob_clean: pd.DataFrame,
            min_age: int = 0, max_age: int = 110) -> pd.DataFrame:
    """Computes Age per-row as YEAR - BirthYear, so age tracks correctly
    across every year a member appears in main_df, instead of one frozen
    snapshot value. Caps implausible ages from bad DOB parses."""
    main_df = main_df.merge(dob_clean[["Member_Key", "BirthYear"]], on="Member_Key", how="left")
    main_df["Age"] = (main_df["YEAR"] - main_df["BirthYear"]).clip(lower=min_age, upper=max_age)
    main_df.drop(columns=["BirthYear"], inplace=True)
    return main_df


# ═══════════════════════════════════════════════════════════════════════════
#  cpt_df
# ═══════════════════════════════════════════════════════════════════════════
def fit_cpt_stats(cpt_train: pd.DataFrame) -> dict:
    cpt = cpt_train.copy()
    cpt["Procedure_Code"] = cpt["Procedure_Code"].fillna("Unknown").astype(str)
    cpt["YEAR"] = _to_year(cpt["StartDate"])
    top_per_group = cpt.groupby(["Member_Key", "YEAR"])["Procedure_Code"].agg(lambda x: x.mode().iloc[0] if len(x) else "Unknown")
    freq = top_per_group.value_counts(normalize=True)
    return {"top_proc_freq": freq.to_dict(), "top_proc_fallback": freq.min() if len(freq) else 0.0}


def aggregate_cpt(cpt: pd.DataFrame, stats: dict) -> pd.DataFrame:
    cpt = cpt.copy()
    cpt["Procedure_Code"] = cpt["Procedure_Code"].fillna("Unknown").astype(str)
    cpt["YEAR"] = _to_year(cpt["StartDate"])  # <-- actual year extraction, not a raw rename

    agg = cpt.groupby(["Member_Key", "YEAR"])["Procedure_Code"].agg(
        cpt_ProcedureCount   ="count",
        cpt_UniqueProcedures ="nunique",
        cpt_TopProcedure     =lambda x: x.mode().iloc[0] if len(x) else "Unknown",
    ).reset_index()

    agg["cpt_ProcedureDiversity"] = agg["cpt_UniqueProcedures"] / agg["cpt_ProcedureCount"]
    # train-only frequency map; unseen top-procedures fall back to the
    # rarest train frequency rather than 0
    agg["cpt_TopProcedure"] = agg["cpt_TopProcedure"].map(stats["top_proc_freq"]).fillna(stats["top_proc_fallback"])

    print(f"    cpt_df  → {agg.shape[0]:,} rows after aggregation")
    return agg


# ═══════════════════════════════════════════════════════════════════════════
#  drg_df
# ═══════════════════════════════════════════════════════════════════════════
def fit_drg_stats(drg_train: pd.DataFrame) -> dict:
    drg = drg_train.copy()
    drg["Code"] = drg["Code"].fillna("Unknown").astype(str)

    # "rare" is judged from TRAIN ONLY, so it doesn't depend on whether a
    # code happens to reappear in the 2024 holdout
    code_freq = drg["Code"].value_counts()
    rare_codes = set(code_freq[code_freq == 1].index)

    drg["YEAR"] = _to_year(drg["Start_Date_Year"])
    most_common_per_group = drg.groupby(["Member_Key", "YEAR"])["Code"].agg(lambda x: x.mode().iloc[0] if len(x) else "Unknown")
    mc_freq = most_common_per_group.value_counts(normalize=True)

    return {"rare_codes": rare_codes, "mostcommon_freq": mc_freq.to_dict(),
            "mostcommon_fallback": mc_freq.min() if len(mc_freq) else 0.0
           }


def aggregate_drg(drg: pd.DataFrame, stats: dict) -> pd.DataFrame:
    drg = drg.copy()
    drg["Code"] = drg["Code"].fillna("Unknown").astype(str)
    drg["YEAR"] = _to_year(drg["Start_Date_Year"])

    agg = drg.groupby(["Member_Key", "YEAR"])["Code"].agg( drg_ClaimsCount="count",drg_UniqueDRGs ="nunique",
                                                          drg_MostCommon =lambda x: x.mode().iloc[0] if len(x) else "Unknown"
                                                         ).reset_index()

    # rarity flag uses the TRAIN-derived rare_codes set -- fixed, doesn't
    # shift depending on which split you're scoring
    drg["is_rare"] = drg["Code"].isin(stats["rare_codes"]).astype(int)
    rare_agg = drg.groupby(["Member_Key", "YEAR"])["is_rare"].sum().reset_index()
    rare_agg.rename(columns={"is_rare": "drg_RareDRGCount"}, inplace=True)
    agg = agg.merge(rare_agg, on=["Member_Key", "YEAR"], how="left")

    agg["drg_MostCommon"] = agg["drg_MostCommon"].map(stats["mostcommon_freq"]).fillna(stats["mostcommon_fallback"])

    print(f"    drg_df  → {agg.shape[0]:,} rows after aggregation")
    return agg


# ═══════════════════════════════════════════════════════════════════════════
#  icd_df
# ═══════════════════════════════════════════════════════════════════════════
# NOTE: verify these prefixes against your actual code system before trusting
# icd_ChronicDiagnosisCount -- this assumes every Diagnosis_Code is ICD-10.
# If any years in your panel predate the ICD-10 transition (or weren't
# remapped), this flag will silently under-count chronic conditions for
# those years, since ICD-9 codes (e.g. "250.xx" for diabetes) don't share
# these prefixes at all. "M79" (unspecified soft-tissue disorder) also looks
# out of place next to the rest of this list -- double check it's intentional.


def fit_icd_stats(icd_train: pd.DataFrame) -> dict:
    icd = icd_train.copy()
    icd["Diagnosis_Code"] = (
        icd["Diagnosis_Code"].fillna("Unknown").astype(str).str.upper().str.strip()
    )

    # Safely parse year whether it's an integer or string
    if pd.api.types.is_numeric_dtype(icd["Start_Date"]):
        icd["YEAR"] = icd["Start_Date"].astype(int)
    else:
        icd["YEAR"] = (
            pd.to_datetime(icd["Start_Date"], errors="coerce")
            .dt.year.fillna(0)
            .astype(int)
        )

    # --- DYNAMIC CHRONIC SELECTION ---
    # Extract the 3-character category (e.g., 'E11' from 'E11.9')
    icd["ICD_Prefix"] = icd["Diagnosis_Code"].str[:3]

    # Let's say you want to dynamically flag the top 11 most frequent prefixes as chronic categories
    top_prefixes = icd["ICD_Prefix"].value_counts().head(11).index.tolist()
    chronic_tuple = tuple(top_prefixes)

    # Optimized mode calculation using value_counts grouped by levels
    top_per_group = (
        icd.groupby(["Member_Key", "YEAR"])["Diagnosis_Code"]
        .value_counts()
        .groupby(level=[0, 1])
        .idxmax()
        .apply(lambda x: x[2])
    )

    freq = top_per_group.value_counts(normalize=True)

    return {
        "top_icd_freq": freq.to_dict(),
        "top_icd_fallback": freq.min() if len(freq) else 0.0,
        "chronic_prefixes": chronic_tuple,  # Stored safely in stats dictionary
    }


def aggregate_icd(icd: pd.DataFrame, stats: dict) -> pd.DataFrame:
    icd = icd.copy()
    icd["Diagnosis_Code"] = (
        icd["Diagnosis_Code"].fillna("Unknown").astype(str).str.upper().str.strip()
    )

    # Retrive chronic definitions calculated during training (No Leakage!)
    target_prefixes = stats.get("chronic_prefixes", ())

    # Flag conditions matching our target list
    icd["is_chronic"] = (
        icd["Diagnosis_Code"].str.startswith(target_prefixes).astype(int)
    )

    if pd.api.types.is_numeric_dtype(icd["Start_Date"]):
        icd["YEAR"] = icd["Start_Date"].astype(int)
    else:
        icd["YEAR"] = (
            pd.to_datetime(icd["Start_Date"], errors="coerce")
            .dt.year.fillna(0)
            .astype(int)
        )

    # Efficient vector aggregations
    agg = icd.groupby(["Member_Key", "YEAR"]).agg(
        icd_DiagnosisCount=("Diagnosis_Code", "count"),
        icd_UniqueDiagnoses=("Diagnosis_Code", "nunique"),
        icd_ChronicDiagnosisCount=("is_chronic", "sum"),
    )

    # Fast group-wise mode extraction replacing the slow lambda loop
    top_icd = (
        icd.groupby(["Member_Key", "YEAR"])["Diagnosis_Code"]
        .value_counts()
        .groupby(level=[0, 1])
        .idxmax()
        .apply(lambda x: x[2])
    )

    agg["icd_TopICD"] = top_icd
    agg = agg.reset_index()

    # Apply mappings and calculate density ratios
    agg["icd_ICDDiversity"] = (
        agg["icd_UniqueDiagnoses"] / agg["icd_DiagnosisCount"]
    )
    agg["icd_TopICD"] = (
        agg["icd_TopICD"]
        .map(stats["top_icd_freq"])
        .fillna(stats["top_icd_fallback"])
    )

    print(f"    icd_df  → {agg.shape[0]:,} rows after aggregation")
    return agg

# ═══════════════════════════════════════════════════════════════════════════
#  MERGE — combine all aggregated tables into one feature table
# ═══════════════════════════════════════════════════════════════════════════

def build_feature_table( main: pd.DataFrame,dob:  pd.DataFrame, cpt:  pd.DataFrame,
                            drg:  pd.DataFrame, icd:  pd.DataFrame, min_age: int = 0,
                            max_age: int = 110,) -> pd.DataFrame:
    """
    Merges all aggregated sub-tables onto main_df and computes per-row Age.

    Fixes vs. original:
    1. Works on .copy() of every input -> no mutation of caller's DataFrames.
    2. Computes Age = YEAR - BirthYear per row (mirrors add_age() from Snippet 1).
       The original merge silently skipped this, leaving BirthYear as a raw
       column and Age undefined -- an input that looks numeric but means nothing.
    3. Casts Member_Key/YEAR for *all* frames (including dob) in one block.
    """

    # ── 0. Defensive copies ──────────────────────────────────────────────────
    main = main.copy()
    dob  = dob.copy()
    cpt  = cpt.copy()
    drg  = drg.copy()
    icd  = icd.copy()

    # ── 1. Coerce join keys to consistent dtypes ─────────────────────────────
    for df in [main, dob, cpt, drg, icd]:
        df["Member_Key"] = df["Member_Key"].astype(int)

    for df in [main, cpt, drg, icd]:          # dob has no YEAR dimension
        if "YEAR" in df.columns:
            df["YEAR"] = df["YEAR"].astype(int)

    # ── 2. Sequential left-joins ─────────────────────────────────────────────
    # dob: one row per member, joined on Member_Key only
    merged = main.merge(dob,  on="Member_Key",           how="left")
    merged = merged.merge(cpt, on=["Member_Key", "YEAR"], how="left")
    merged = merged.merge(drg, on=["Member_Key", "YEAR"], how="left")
    merged = merged.merge(icd, on=["Member_Key", "YEAR"], how="left")

    # ── 3. Per-row age computation (fix #2) ──────────────────────────────────
    # BirthYear comes from aggregate_dob(); Age must vary across YEAR rows,
    # so it cannot be pre-computed in process_dob as a single frozen value.
    if "BirthYear" in merged.columns and "YEAR" in merged.columns:
        merged["Age"] = (
            (merged["YEAR"] - merged["BirthYear"])
            .clip(lower=min_age, upper=max_age)
        )
        merged.drop(columns=["BirthYear"], inplace=True)

    # ── 4. Zero-fill aggregated event columns (no event = 0, not NaN) ────────
    agg_fill = [c for c in merged.columns if c.startswith(("cpt_", "drg_", "icd_"))]
    merged[agg_fill] = merged[agg_fill].fillna(0)

    print(f"    merged  → {merged.shape[0]:,} rows × {merged.shape[1]} cols")
    return merged



# ─────────────────────────────────────────────────────────────────────────────
#  Label engineering + temporal split
# ─────────────────────────────────────────────────────────────────────────────

def prepare_splits(
    feature_table: pd.DataFrame,
    cost_col:     str   = "NEXT_YEAR_COST",
    threshold:    float = 30_000.0,
    val_year:     int   = 2023,
    extra_drop:   Optional[List[str]] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series]:
    
    """
    Returns (X_train, X_test, X_val, y_train, y_test, y_val).

    Design decisions
    ────────────────
    • Time-based split on YEAR (preferred over stratified random split when a
      temporal dimension exists — prevents future data from leaking into train).
    • YEAR assignment:
        train  → YEAR < val_year           (historical context)
        val    → YEAR == val_year          (hyper-parameter tuning)
        test   → YEAR == test_year         (final holdout)
      Rows where YEAR > test_year are dropped (no known NEXT_YEAR_COST yet).
    • HighCostLabel is built BEFORE splitting so the threshold is applied
      uniformly; both HighCostLabel and cost_col are then excluded from X.
    • Rows with missing NEXT_YEAR_COST (e.g. member's last year in panel,
      no following year to look up) are dropped — they carry no valid label.
    """

    if "YEAR" not in feature_table.columns:
        raise ValueError("YEAR column is required for temporal integrity split.")

    df = feature_table.copy()

    # ── 1. Drop rows with undefined next-year cost ────────────────────────────
    n_before = len(df)
    df = df.dropna(subset=[cost_col]).reset_index(drop=True)
    dropped = n_before - len(df)
    if dropped:
        print(f"    Dropped {dropped:,} rows with missing {cost_col}.")

    # ── 2. Engineer binary target ─────────────────────────────────────────────
    df["HighCostLabel"] = (df[cost_col] > threshold).astype(np.int8)

    # ── 3. Identify feature columns (no leakage) ─────────────────────────────
    always_drop = {cost_col, "HighCostLabel"}
    if extra_drop:
        always_drop.update(extra_drop)
    feature_cols = [c for c in df.columns if c not in always_drop]

    # ── 4. Temporal masks ─────────────────────────────────────────────────────
    train_mask = df["YEAR"] <  val_year
    val_mask   = df["YEAR"] == val_year

    # Warn if any years fall outside all three windows
    unassigned = df.loc[~(train_mask | val_mask), "YEAR"].unique()
    if len(unassigned):
        print(f"    Warning: {len(unassigned)} YEAR value(s) not assigned to any split: {unassigned}")

    # ── 5. Build splits ───────────────────────────────────────────────────────
    def _split(mask):
        X = df.loc[mask, feature_cols].reset_index(drop=True)
        y = df.loc[mask, "HighCostLabel"].reset_index(drop=True)
        return X, y

    X_train, y_train = _split(train_mask)
    X_val,   y_val   = _split(val_mask)

    # ── 6. Sanity report ─────────────────────────────────────────────────────
    print(f"\n    {'Split':<8} {'Rows':>8}  {'HighCost N':>12}  {'HighCost %':>10}")
    print(f"    {'-'*44}")
    for name, y in [("train", y_train), ("val", y_val)]:
        pos = int(y.sum())
        pct = pos / max(len(y), 1) * 100
        print(f"    {name:<8} {len(y):>8,}  {pos:>12,}  {pct:>9.1f}%")
    print(f"    Features: {len(feature_cols)} columns\n")

    return X_train, X_val, y_train, y_val


In [ ]:

# ─────────────────────────────────────────────────────────────────────────────
#  End-to-end usage (wires together everything above)
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    BASE_DIR = r"/kaggle/input/datasets/haseebekhan/healthcare-cost-dataset/Healthcare Cost Dataset"
    
    TRAINING_PATH = os.path.join(BASE_DIR, "train", "train") + os.sep
    TESTING_PATH  = os.path.join(BASE_DIR, "test",  "test")  + os.sep

    # ── Load raw tables ───────────────────────────────────────────────────────
    main_train = pd.read_csv(TRAINING_PATH + "main_df_train.csv")
    main_test  = pd.read_csv(TESTING_PATH  + "main_df_test.csv")
    dob_train  = pd.read_csv(TRAINING_PATH + "dob_df.csv")
    dob_test   = pd.read_csv(TESTING_PATH  + "dob_df.csv")
    cpt_train  = pd.read_csv(TRAINING_PATH + "cpt_df_train.csv")
    cpt_test   = pd.read_csv(TESTING_PATH  + "cpt_df_test.csv")
    drg_train  = pd.read_csv(TRAINING_PATH + "drg_df_train.csv")
    drg_test   = pd.read_csv(TESTING_PATH  + "drg_df_test.csv")
    icd_train  = pd.read_csv(TRAINING_PATH + "icd_df_train.csv")
    icd_test   = pd.read_csv(TESTING_PATH  + "icd_df_test.csv")

    # ── Fit stats on TRAIN ONLY ───────────────────────────────────────────────
    main_stats = fit_main_stats(main_train)
    dob_stats  = fit_dob_stats(dob_train)
    cpt_stats  = fit_cpt_stats(cpt_train)
    drg_stats  = fit_drg_stats(drg_train)
    icd_stats  = fit_icd_stats(icd_train)

    # ── Aggregate (apply stats to BOTH splits) ────────────────────────────────
    main_tr = aggregate_main(main_train, main_stats)
    main_te = aggregate_main(main_test,  main_stats)

    dob_tr  = aggregate_dob(dob_train, dob_stats)
    dob_te  = aggregate_dob(dob_test,  dob_stats)

    cpt_tr  = aggregate_cpt(cpt_train, cpt_stats)
    cpt_te  = aggregate_cpt(cpt_test,  cpt_stats)

    drg_tr  = aggregate_drg(drg_train, drg_stats)
    drg_te  = aggregate_drg(drg_test,  drg_stats)

    icd_tr  = aggregate_icd(icd_train, icd_stats)
    icd_te  = aggregate_icd(icd_test,  icd_stats)

    # ── Build feature tables ──────────────────────────────────────────────────
    features_train = build_feature_table(main_tr, dob_tr, cpt_tr, drg_tr, icd_tr)
    features_test  = build_feature_table(main_te, dob_te, cpt_te, drg_te, icd_te)

    # ── Stack train+test into one panel, then split temporally ───────────────
    # (val_year=2022 is held out of training; test_year=2023 is the holdout)
       # Use only training years for train/val — the labeled split.
    # First check what years actually exist in features_train (per the YEAR check
    # from earlier), then pick val_year as the LAST labeled training year.

    X_train, X_val, y_train, y_val = prepare_splits(
    features_train,
    cost_col   = "NEXT_YEAR_COST",
    threshold  = 30_000.0,
    val_year   = 2023,
    )
    assert len(y_train) > 0 and len(y_val) > 0, "a split is empty — check the YEAR values"
    
    submission_cols = [c for c in features_test.columns if c not in ("NEXT_YEAR_COST", "HighCostLabel")]
    X_submission = features_test[submission_cols]
    
    # ── Multicollinearity pruning (fit on TRAIN ONLY) ─────────────────────────
    ID_COLS = ["Member_Key"]   # identifiers, never real predictors -- exclude from VIF entirely
    vif_drop_cols = fit_vif_drop_list(X_train, exclude_cols=ID_COLS, vif_threshold=10.0)
    
    X_train_lr = drop_multicollinear(X_train, vif_drop_cols)
    X_val_lr   = drop_multicollinear(X_val,   vif_drop_cols)
    X_sub_lr   = drop_multicollinear(X_submission, vif_drop_cols)
    # features_test (2024) has no NEXT_YEAR_COST — it's not for prepare_splits.
    # It's your submission set: predict on it, don't try to score it locally.
    # submission_cols = [c for c in features_test.columns if c not in ("NEXT_YEAR_COST", "HighCostLabel")]
    # X_submission = features_test[submission_cols]
    # # preds = model.predict(X_submission)  →  this is what you submit to the leaderboard

In [ ]:

from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. Select only numeric columns
numeric_df = X_train_lr.select_dtypes(include=['number']).copy()

# 2. Replace infinity values (inf and -inf) with NaN
numeric_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 3. Drop columns that are completely empty/NaN (mean calculation would fail on these)
numeric_df.dropna(axis=1, how='all', inplace=True)

# 4. Fill remaining NaNs with the column mean
numeric_df = numeric_df.fillna(numeric_df.mean())

# 5. Quick safety check: Drop columns with zero variance (all identical values)
# Because an entirely constant column will cause perfect collinearity issues
numeric_df = numeric_df.loc[:, numeric_df.nunique() > 1]

# 6. Add the constant intercept for VIF
numeric_df['intercept'] = 1

# 7. Calculate VIF safely
vif_data = pd.DataFrame()
vif_data["feature"] = [col for col in numeric_df.columns if col != 'intercept']
vif_data["VIF"] = [
    variance_inflation_factor(numeric_df.values, i) 
    for i in range(len(numeric_df.columns)) 
    if numeric_df.columns[i] != 'intercept'
]

with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(vif_data.sort_values(by="VIF", ascending=False))

In [ ]:

from statsmodels.stats.outliers_influence import variance_inflation_factor

# 1. Select only numeric columns
numeric_df = X_train.select_dtypes(include=['number']).copy()

# 2. Replace infinity values (inf and -inf) with NaN
numeric_df.replace([np.inf, -np.inf], np.nan, inplace=True)

# 3. Drop columns that are completely empty/NaN (mean calculation would fail on these)
numeric_df.dropna(axis=1, how='all', inplace=True)

# 4. Fill remaining NaNs with the column mean
numeric_df = numeric_df.fillna(numeric_df.mean())

# 5. Quick safety check: Drop columns with zero variance (all identical values)
# Because an entirely constant column will cause perfect collinearity issues
numeric_df = numeric_df.loc[:, numeric_df.nunique() > 1]

# 6. Add the constant intercept for VIF
numeric_df['intercept'] = 1

# 7. Calculate VIF safely
vif_data = pd.DataFrame()
vif_data["feature"] = [col for col in numeric_df.columns if col != 'intercept']
vif_data["VIF"] = [
    variance_inflation_factor(numeric_df.values, i) 
    for i in range(len(numeric_df.columns)) 
    if numeric_df.columns[i] != 'intercept'
]

print(vif_data.sort_values(by="VIF", ascending=False))

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(vif_data.sort_values(by="VIF", ascending=False))

In [ ]:
import seaborn as sns

# 1. Compute the correlation matrix
corr_matrix = X_train_lr.select_dtypes(include=['number']).corr()

# 2. Plot it as a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix to Spot Multicollinerity")
plt.show()

In [ ]:
# 1. Compute the correlation matrix
corr_matrix = X_train.select_dtypes(include=['number']).corr()

# 2. Plot it as a heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5)
plt.title("Correlation Matrix to Spot Multicollinerity")
plt.show()

# **Corelation Columns:**

In [ ]:
# 1. Filter cost_cols to only include columns that actually exist in the DataFrame
existing_cost_cols = [col for col in stats["cost_cols"] if col in train_clean.columns]

# 2. Melt the matrix to find top relationships using ONLY existing columns
c = train_clean[existing_cost_cols].corr().abs()
s = c.unstack()
so = s.sort_values(kind="quicksort", ascending=False)

# 3. Print pairs that have a correlation between 0.70 and 0.99
print(so[(so > 0.70) & (so < 1.0)].drop_duplicates().head(20))

In [ ]:
print(list(df[all_cols].columns))